# PostgreSQL Silver — Transformation

## Source

Bronze PostgreSQL datasets:

- users
- subscription
- payments

## Purpose

Transform the PostgreSQL Bronze data into technically consistent Silver datasets.

All transformation decisions in this notebook are based on the PostgreSQL EDA findings.

## Transformation approach

The Silver layer will:

- preserve valid identifiers and relationships
- apply safe technical cleaning
- preserve intentional dirty data where it represents useful source information
- flag detected data-quality problems instead of silently deleting records
- avoid changing the business meaning of source records
- derive only useful analytical fields

## Confirmed EDA findings

### Users

- 10,000 records
- Primary IDs are unique
- One missing email
- One invalid country value
- One email contains surrounding whitespace
- One last_login occurs before created_at
- 9,999 users have no login value
- No relationship problems

### Subscriptions

- 20,032 records
- Primary IDs are unique
- No null values
- One end_date occurs before start_date
- User → Subscription relationship is intact
- Each user has between 1 and 3 subscriptions

### Payments

- 11,771 records
- Primary IDs are unique
- No null values
- Subscription → Payment relationship is intact
- Payment retries create between 1 and 3 attempts
- Payment amounts are within the generated business domain

## Transformation decisions

### Users

- Preserve user_id unchanged.
- Trim surrounding whitespace from email.
- Preserve NULL email values.
- Normalize country formatting using trim and uppercase only.
- Preserve invalid country values.
- Add validation flags for:
  - missing email
  - invalid country
  - last_login before created_at
- Preserve original timestamps.
- Add date-level columns for analytical use.

### Subscriptions

- Preserve identifiers and relationships unchanged.
- Preserve start_date and end_date timestamps.
- Do not delete or overwrite invalid date chronology.
- Add a validation flag for end_date before start_date.
- Add date-level columns for analytical use.

### Payments

- Preserve identifiers and relationships unchanged.
- Preserve payment meaning and status.
- Normalize amount precision from decimal(38,18) to decimal(10,2).
- Normalize currency and payment_status formatting.
- Preserve payment timestamps.
- Add a date-level payment column.

## Rule

Questionable records remain available in Silver.

Data-quality problems are represented explicitly through validation fields rather than deleting source records.

In [1]:
if "spark" in globals():
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


spark = (
    SparkSession.builder
    .appName("postgres_silver_transformation")
    .master("spark://spark-master:7077")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.autoBroadcastJoinThreshold", "10m")
    .config(
        "spark.sql.parquet.enableVectorizedReader",
        "true",
    )
    .config(
        "spark.sql.legacy.parquet.nanosAsLong",
        "true",
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000",
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minio",
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minio123",
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true",
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem",
    )
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 14:02:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/06 14:02:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
POSTGRES_BRONZE_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "postgres/ingestion_date=2026-09-06-Sep"
)


POSTGRES_SILVER_PATH = (
    "s3a://end-to-end-streaming-data-platform-silver/"
    "postgres"
)

In [4]:
df_users_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/users.parquet"
)

df_subscriptions_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/subscription.parquet"
)

df_payments_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/payments.parquet"
)

26/09/06 14:02:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Users Transformation

In [5]:
VALID_COUNTRIES = (
    "KSA",
    "EGY",
    "UAE",
    "QAT",
    "KWT",
    "JOR",
    "MAR",
)


df_users_silver = (
    df_users_bronze
    .withColumn(
        "email",
        F.when(
            F.col("email").isNotNull(),
            F.trim(F.col("email")),
        )
    )
    .withColumn(
        "country",
        F.upper(
            F.trim(
                F.col("country")
            )
        )
    )
    .withColumn(
        "is_missing_email",
        F.col("email").isNull()
    )
    .withColumn(
        "is_invalid_country",
        (
            F.col("country").isNull()
            |
            ~F.col("country").isin(
                *VALID_COUNTRIES
            )
        )
    )
    .withColumn(
        "is_login_before_creation",
        (
            F.col("last_login").isNotNull()
            &
            (
                F.col("last_login")
                < F.col("created_at")
            )
        )
    )
    .withColumn(
        "created_date",
        F.to_date("created_at")
    )
    .withColumn(
        "last_login_date",
        F.to_date("last_login")
    )
)

Users Result

In [6]:
print("=" * 70)
print("USERS SILVER")
print("=" * 70)

df_users_silver.printSchema()

df_users_silver.show(
    10,
    truncate=False, vertical=True
)

USERS SILVER
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)
 |-- is_missing_email: boolean (nullable = false)
 |-- is_invalid_country: boolean (nullable = true)
 |-- is_login_before_creation: boolean (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_login_date: date (nullable = true)



-RECORD 0--------------------------------------------------------
 user_id                  | 01a06562-ea04-7fee-a81c-484c4a4020c5 
 email                    | NULL                                 
 country                  | MAR                                  
 account_status           | active                               
 created_at               | 2026-01-01 00:00:00                  
 last_login               | NULL                                 
 is_missing_email         | true                                 
 is_invalid_country       | false                                
 is_login_before_creation | false                                
 created_date             | 2026-01-01                           
 last_login_date          | NULL                                 
-RECORD 1--------------------------------------------------------
 user_id                  | 01a06562-ea07-7588-a5e4-e68207754966 
 email                    | davidlane@example.com                
 country  

Subscriptions Transformation

In [6]:
df_subscriptions_silver = (
    df_subscriptions_bronze
    .withColumn(
        "is_end_date_before_start_date",
        (
            F.col("end_date").isNotNull()
            &
            (
                F.col("end_date")
                < F.col("start_date")
            )
        )
    )
    .withColumn(
        "start_date_day",
        F.to_date("start_date")
    )
    .withColumn(
        "end_date_day",
        F.to_date("end_date")
    )
)

Subscriptions Result

In [8]:
print("=" * 70)
print("SUBSCRIPTIONS SILVER")
print("=" * 70)

df_subscriptions_silver.printSchema()

df_subscriptions_silver.show(
    10,
    truncate=False, vertical=True
)

SUBSCRIPTIONS SILVER
root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- is_end_date_before_start_date: boolean (nullable = true)
 |-- start_date_day: date (nullable = true)
 |-- end_date_day: date (nullable = true)

-RECORD 0-------------------------------------------------------------
 subscription_id               | 01a06562-ea05-771e-8f91-7d079a73b5eb 
 user_id                       | 01a06562-ea04-7fee-a81c-484c4a4020c5 
 plan_type                     | free                                 
 start_date                    | 2026-01-01 00:00:00                  
 end_date                      | 2025-12-31 00:00:00                  
 is_active                     | true                                 
 auto_renew                 

Payments Transformation

In [7]:
df_payments_silver = (
    df_payments_bronze
    .withColumn(
        "amount",
        F.col("amount").cast(
            DecimalType(10, 2)
        )
    )
    .withColumn(
        "currency",
        F.upper(
            F.trim(
                F.col("currency")
            )
        )
    )
    .withColumn(
        "payment_status",
        F.lower(
            F.trim(
                F.col("payment_status")
            )
        )
    )
    .withColumn(
        "payment_date_day",
        F.to_date("payment_date")
    )
)

Payments Result

In [10]:
print("=" * 70)
print("PAYMENTS SILVER")
print("=" * 70)

df_payments_silver.printSchema()

df_payments_silver.show(
    10,
    truncate=False, vertical=True
)

PAYMENTS SILVER
root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date_day: date (nullable = true)

-RECORD 0------------------------------------------------
 payment_id       | 01a06562-ea10-737c-b60d-9cb47c616edc 
 subscription_id  | 01a06562-ea0f-7b96-8550-84925d20f499 
 amount           | 9.99                                 
 currency         | USD                                  
 payment_date     | 2026-01-01 02:01:00                  
 payment_status   | success                              
 payment_date_day | 2026-01-01                           
-RECORD 1------------------------------------------------
 payment_id       | 01a06562-ea15-7981-b167-9f4fd59fbe14 
 subscription_id  | 01a06562-ea14-7db9-8e5f-08bae1ed0033 
 amount           | 9.99   

Final Silver Counts

In [11]:
print("=" * 70)
print("POSTGRES SILVER — RECORD COUNTS")
print("=" * 70)

print(
    "Users:",
    df_users_silver.count(),
)

print(
    "Subscriptions:",
    df_subscriptions_silver.count(),
)

print(
    "Payments:",
    df_payments_silver.count(),
)

POSTGRES SILVER — RECORD COUNTS
Users: 10000
Subscriptions: 20033
Payments: 11561


In [8]:
(
    df_users_silver
    .write
    .mode("overwrite")
    .parquet(
        f"{POSTGRES_SILVER_PATH}/users.parquet"
    )
)


(
    df_subscriptions_silver
    .write
    .mode("overwrite")
    .parquet(
        f"{POSTGRES_SILVER_PATH}/subscription.parquet"
    )
)


(
    df_payments_silver
    .write
    .mode("overwrite")
    .parquet(
        f"{POSTGRES_SILVER_PATH}/payments.parquet"
    )
)

In [10]:
df_users_written = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/users.parquet"
)

df_subscriptions_written = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/subscription.parquet"
)

df_payments_written = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/payments.parquet"
)


print("=" * 70)
print("POSTGRES SILVER — WRITE VERIFICATION")
print("=" * 70)

print(
    "Users:",
    df_users_written.count(),
)

print(
    "Subscriptions:",
    df_subscriptions_written.count(),
)

print(
    "Payments:",
    df_payments_written.count(),
)

POSTGRES SILVER — WRITE VERIFICATION
Users: 10000
Subscriptions: 20032
Payments: 11771


# Transformation Summary

## Datasets transformed

- users
- subscription
- payments

## Applied transformations

### Users

- Trimmed surrounding whitespace from email values.
- Preserved missing email values.
- Normalized country formatting.
- Added:
  - is_missing_email
  - is_invalid_country
  - is_login_before_creation
- Added analytical date columns.

### Subscriptions

- Preserved identifiers and source relationships.
- Preserved source timestamps.
- Added:
  - is_end_date_before_start_date
- Added analytical date columns.

### Payments

- Preserved identifiers and source relationships.
- Normalized monetary precision.
- Normalized currency formatting.
- Normalized payment status formatting.
- Added payment_date_day.

## Dirty data handling

Intentional dirty records were not deleted.

The Silver layer preserves the underlying source records and exposes detected quality problems through validation fields.

## Relationship preservation

The transformation does not modify:

- user_id relationships
- subscription_id relationships
- User → Subscription relationships
- Subscription → Payment relationships

## Output

Three Silver datasets were written:

- users.parquet
- subscription.parquet
- payments.parquet

The PostgreSQL Bronze → Silver transformation is complete.

The next stage is PostgreSQL Silver testing.